In [9]:
import os
import sys
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
if not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.lines as mlines
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cmocean
CMAP = cmocean.cm.balance
import geopandas as gpd

In [10]:
# =============================================================================
# Eddies
# =============================================================================
EDDIES = {
    # Cyclone: crossed straight through (near) the center -> strategy "crossing".
    # survey = window searched for SADCP crossings and LADCP stations (can be the whole cruise).
    # The candidate passes are printed; crossing_pick chooses among them.
    "CE18": dict(
        name="CE18_20260523", eddy_type=-1, strategy="crossing",
        survey=dict(t0="2026-06-28T00:00", t1="2026-07-27T00:00"),
        crossing_pick="first",  # "first", "closest", or the id of a pass in the printed table
        crossing=None,      # dict(t0=..., t1=...) to impose the crossing instead of detecting it
        r_limit_km=None,    # LADCP stations kept within this distance (default LADCP_R_FACTOR x Rmax)
    ),
    # Anticyclone: never crossed through the center -> radial decomposition (to be revised)
    "AE11": dict(
        name="AE11_20260514", eddy_type=1, strategy="radial",
        sections={
            # stations 005-048: 27 to 186 km from the center, all around the eddy
            "radial_0701_0710": dict(mode="radial", t0="2026-07-01T00:00", t1="2026-07-11T00:00",
                                     r_limit_km=220),
        }),
}
RUN = ["CE18"]          # eddies to process


In [11]:
SADCP_AVG = "10min"          # SADCP time averaging (~3 km at 10 kn)
USE_DETIDED = True           # use UVEL_ADCP_CORTIDE / VVEL_ADCP_CORTIDE when present
Z_SPLIT = 150.0              # m: OS150 above, OS38 below
ZMAX_SADCP = 1000.0
ZMAX_LADCP = 1500.0
DS_KM = 3.0                  # bin size along a section (SADCP)
R_BIN_KM = 10.0              # radius bin (radial mode, SADCP)
LADCP_MAX_GAP_KM = 60.0      # do not interpolate between LADCP stations further apart
FIT_CENTER = True            # radial mode: refine the TOEddies center from SADCP
FIT_LAYER = (30.0, 200.0)    # m: layer used to fit the center
FIT_SEARCH_KM = 60.0         # search +- this offset around the TOEddies center
CENTER_SMOOTH_DAYS = 3       # running mean on the daily TOEddies centers (they move on a 1/8° grid)
PROFILE_DEPTHS = [50, 100, 200, 400, 700]   # radial mode: v_theta(r) at these depths
R_KM = 6371.0
crs = ccrs.PlateCarree()


In [12]:
# =============================================================================
# Paths
# =============================================================================
BASE = Path("/Users/emmagurcan/Documents/France/ENS/Speich/WHIRLS")
DIR_SADCP = BASE / "data/SADCP"
SADCP_FILES = {"OS150": DIR_SADCP / "WHIRLS_NB150_TOT_osite.nc",   # full cruise (README)
               "OS38": DIR_SADCP / "WHIRLS_BB38_TOT_osite.nc"}
DIR_LADCP = BASE / "data/LADCP/traitement_avec_SADCP"
DIR_SSH = BASE / "data/SSH"
DIR_EDDIES = {"TOEDDIES_better_ADT": BASE / "data/TOEDDIES/TOEDDIES_better_ADT",
              "TOEDDIES": BASE / "data/TOEDDIES/TOEDDIES"}
FNAME_BATHY = BASE / "data/bathymetry/etopo2.nc"
DIR_TRACKS = BASE / "figures"
DIR_CODE = BASE / "analysis"          # where ladcp_tools.py lives
DIR_EDDIES_BETTER = DIR_EDDIES["TOEDDIES_better_ADT"]
DIR_EDDIES_STD = DIR_EDDIES["TOEDDIES"]
OUT_DIR = BASE / "figures2/"
SADCP_TRACK_CSV = BASE / "figures/sadcp_vs_geostrophy/sadcp_vs_geostrophy_track.csv"

In [13]:

# =============================================================================
# Pure geometry (no I/O)
# =============================================================================
def to_xy(lon, lat, lon0, lat0):
    """Local tangent plane, km."""
    x = np.deg2rad(np.asarray(lon, float) - lon0) * R_KM * np.cos(np.deg2rad(lat0))
    y = np.deg2rad(np.asarray(lat, float) - lat0) * R_KM
    return x, y


def fit_line(x, y):
    """Principal axis of the points. Returns origin, along unit vector (oriented in the
    direction of travel) and normal unit vector (to the LEFT of travel)."""
    ok = np.isfinite(x) & np.isfinite(y)
    xm, ym = x[ok].mean(), y[ok].mean()
    w, vec = np.linalg.eigh(np.cov(np.vstack([x[ok] - xm, y[ok] - ym])))
    es = vec[:, np.argmax(w)]
    xi, yi = x[ok][[0, -1]], y[ok][[0, -1]]
    if (xi[1] - xi[0]) * es[0] + (yi[1] - yi[0]) * es[1] < 0:
        es = -es
    en = np.array([-es[1], es[0]])
    return np.array([xm, ym]), es, en


def polar(x, y, u, v):
    """Radius (km), azimuthal (+ anticlockwise) and radial (+ outward) velocity.
    x, y: (n,) ; u, v: (n,) or (n, nz)."""
    r = np.hypot(x, y)
    rr = np.where(r > 0, r, np.nan)
    if np.ndim(u) == 2:
        x, y, rr = x[:, None], y[:, None], rr[:, None]
    vt = (-u * y + v * x) / rr
    vr = (u * x + v * y) / rr
    return r, vt, vr


def fit_center(x, y, u, v, r_max, search=FIT_SEARCH_KM, step=2.0, r_min=10.0, nmin=20):
    """Offset (dx, dy) of the center minimising the fraction of radial kinetic energy."""
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(u) & np.isfinite(v)
    x, y, u, v = x[ok], y[ok], u[ok], v[ok]
    best = (np.inf, 0.0, 0.0)
    for dx in np.arange(-search, search + step / 2, step):
        for dy in np.arange(-search, search + step / 2, step):
            r, vt, vr = polar(x - dx, y - dy, u, v)
            m = (r >= r_min) & (r <= r_max)
            if m.sum() < nmin:
                continue
            cost = np.sum(vr[m] ** 2) / np.sum(vr[m] ** 2 + vt[m] ** 2)
            if cost < best[0]:
                best = (cost, dx, dy)
    return best[1], best[2], best[0]


def bin_columns(coord, vals, edges, nmin=1):
    """Mean of vals (n, nz) in bins of coord. Returns centers, (nbins, nz) mean, counts."""
    idx = np.digitize(coord, edges) - 1
    nb = len(edges) - 1
    out = np.full((nb, vals.shape[1]), np.nan)
    cnt = np.zeros(nb, int)
    for b in range(nb):
        m = idx == b
        cnt[b] = m.sum()
        if cnt[b] >= nmin:
            with np.errstate(all="ignore"):
                out[b] = np.nanmean(vals[m], axis=0)
    return 0.5 * (edges[:-1] + edges[1:]), out, cnt


def interp_columns(coord, vals, grid, max_gap):
    """Linear interpolation of station profiles (n, nz) onto grid, not across gaps."""
    o = np.argsort(coord)
    c, V = np.asarray(coord)[o], vals[o]
    out = np.full((len(grid), V.shape[1]), np.nan)
    for k in range(V.shape[1]):
        m = np.isfinite(V[:, k])
        if m.sum() >= 2:
            cc, vv = c[m], V[m, k]
            col = np.interp(grid, cc, vv, left=np.nan, right=np.nan)
            j = np.clip(np.searchsorted(cc, grid), 1, len(cc) - 1)
            near = np.min(np.abs(grid[:, None] - cc[None, :]), axis=1) <= np.diff(grid).mean() / 2
            col[((cc[j] - cc[j - 1]) > max_gap) & ~near] = np.nan
            out[:, k] = col
        elif m.sum() == 1:
            out[np.argmin(np.abs(grid - c[m][0])), k] = V[m, k][0]
    return out


# =============================================================================
# Data loading
# =============================================================================
def _find(ds, names, required=True):
    for n in names:
        if n in ds.variables:
            return n
    low = {k.lower(): k for k in ds.variables}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    if required:
        raise KeyError(f"None of {names} in {list(ds.variables)}")
    return None


_SADCP_CACHE = {}


def load_sadcp_raw(path):
    """Full-resolution SADCP as Dataset(time, z) with u, v, lat, lon (QC flag == 1)."""
    if path in _SADCP_CACHE:
        return _SADCP_CACHE[path]
    ds = xr.open_dataset(path)
    tn = _find(ds, ["JULD_ADCP", "TIME", "time", "JULD"])
    zn = _find(ds, ["DEPH", "DEPTH", "depth"])
    un = _find(ds, ["UVEL_ADCP_CORTIDE"], required=False) if USE_DETIDED else None
    vn = _find(ds, ["VVEL_ADCP_CORTIDE"], required=False) if USE_DETIDED else None
    if un is None or vn is None:
        un, vn = _find(ds, ["UVEL_ADCP"]), _find(ds, ["VVEL_ADCP"])
    fn = _find(ds, ["CAS_CURRENT_FLAG"], required=False)
    print(f"  {Path(path).name}: using {un}, {vn}")
    tdim = ds[tn].dims[0]
    U = ds[un].squeeze(drop=True)
    zdim = [d for d in U.dims if d != tdim][0]
    u = U.transpose(tdim, zdim).values.astype(float)
    v = ds[vn].squeeze(drop=True).transpose(tdim, zdim).values.astype(float)
    traw = ds[tn].values
    time = (pd.to_datetime(traw) if np.issubdtype(traw.dtype, np.datetime64)
            else pd.Timestamp("1950-01-01") + pd.to_timedelta(traw.astype(float), unit="D"))
    if fn is not None:
        fl = ds[fn].squeeze(drop=True)
        fl = fl.transpose(tdim, zdim).values if zdim in fl.dims else fl.values[:, None]
        good = (fl == 1) if np.issubdtype(fl.dtype, np.number) else \
            np.char.strip(np.char.decode(fl.astype("S"), "ascii", "ignore")) == "1"
        good = np.broadcast_to(good, u.shape)
        u, v = np.where(good, u, np.nan), np.where(good, v, np.nan)
    u[np.abs(u) > 5] = np.nan
    v[np.abs(v) > 5] = np.nan
    zda = ds[zn].squeeze(drop=True)
    z = np.abs(np.nanmedian(zda.transpose(tdim, zdim).values, axis=0) if zda.ndim == 2
               else zda.values).astype(float)
    keepz = np.isfinite(z)
    lat = ds[_find(ds, ["LATITUDE", "lat"])].squeeze(drop=True).values.astype(float)
    lon = ds[_find(ds, ["LONGITUDE", "lon"])].squeeze(drop=True).values.astype(float)
    lat = np.where(np.abs(lat) <= 90, lat, np.nan)
    lon = np.where(np.abs(lon) <= 360, lon, np.nan)
    lon = np.where(lon > 180, lon - 360, lon)
    out = xr.Dataset({"u": (("time", "z"), u[:, keepz]), "v": (("time", "z"), v[:, keepz]),
                      "lat": ("time", lat), "lon": ("time", lon)},
                     coords={"time": np.asarray(time, "datetime64[ns]"), "z": z[keepz]})
    out = out.isel(time=np.asarray(pd.notna(time))).sortby("time").sortby("z")
    out = out.isel(time=~pd.Index(out.time.values).duplicated())
    ds.close()
    _SADCP_CACHE[path] = out
    return out


def sadcp_window(t0, t1):
    """OS150 above Z_SPLIT + OS38 below, averaged over SADCP_AVG, inside [t0, t1]."""
    parts = []
    for inst, f in SADCP_FILES.items():
        if not f.exists():
            print(f"  ! missing {f}")
            continue
        d = load_sadcp_raw(f).sel(time=slice(t0, t1))
        d = d.resample(time=SADCP_AVG).mean()
        zz = d.z.values
        keep = zz <= Z_SPLIT if inst == "OS150" else (zz > Z_SPLIT) & (zz <= ZMAX_SADCP)
        d = d.isel(z=np.where(keep)[0])
        parts.append(d)
    if not parts:
        raise FileNotFoundError("no SADCP file found")
    base = parts[0]
    out = base
    for p in parts[1:]:
        p = p.reindex(time=base.time)
        out = xr.Dataset({"u": xr.concat([out.u, p.u], "z"), "v": xr.concat([out.v, p.v], "z"),
                          "lat": out.lat.fillna(p.lat), "lon": out.lon.fillna(p.lon)})
    out = out.sortby("z")
    ok = np.isfinite(out.lat.values) & np.isfinite(out.lon.values) & np.isfinite(out.u.values).any(1)
    return out.isel(time=ok)


def load_ladcp(t0, t1):
    """Stations cast in [t0, t1]: list of dicts with station, time, lon, lat, z, u, v."""
    sys.path.insert(0, str(DIR_CODE))
    from ladcp_tools import load_all_stations, read_lad
    st = load_all_stations(str(DIR_LADCP) + "/")
    tcol = "datetime" if "datetime" in st.columns else st.select_dtypes("datetime").columns[0]
    st = st[(st[tcol] >= pd.Timestamp(t0)) & (st[tcol] <= pd.Timestamp(t1))]
    out = []
    for _, s in st.iterrows():
        f = s["file"] if "file" in s and str(s["file"]).endswith(".lad") else \
            os.path.join(str(DIR_LADCP), str(s.station), f"{s.station}.lad")
        try:
            _, df = read_lad(f)
        except Exception as e:
            print(f"  ! LADCP {s.station}: {e}")
            continue
        cols = {c.lower(): c for c in df.columns}
        zc = next((cols[c] for c in ("z", "depth", "dep", "pressure", "p") if c in cols), None)
        uc = next((cols[c] for c in ("u", "u_mean", "uvel") if c in cols), None)
        vc = next((cols[c] for c in ("v", "v_mean", "vvel") if c in cols), None)
        if None in (zc, uc, vc):
            raise KeyError(f"LADCP columns not recognised: {list(df.columns)} "
                           "-> edit the names in load_ladcp()")
        df = df[[zc, uc, vc]].astype(float).dropna()
        out.append(dict(station=str(s.station), time=pd.Timestamp(s[tcol]), lon=float(s.lon),
                        lat=float(s.lat), z=np.abs(df[zc].values), u=df[uc].values,
                        v=df[vc].values))
    print(f"  LADCP: {len(out)} station(s) in the window: {', '.join(o['station'] for o in out)}")
    return out


def ladcp_on_grid(stations, zgrid):
    """(n_station, nz) u and v on a common depth grid."""
    U = np.full((len(stations), len(zgrid)), np.nan)
    V = np.full_like(U, np.nan)
    for i, s in enumerate(stations):
        o = np.argsort(s["z"])
        U[i] = np.interp(zgrid, s["z"][o], s["u"][o], left=np.nan, right=np.nan)
        V[i] = np.interp(zgrid, s["z"][o], s["v"][o], left=np.nan, right=np.nan)
    return U, V


def load_track(name):
    f = next(iter(sorted(DIR_TRACKS.rglob(f"tracked_eddy_{name}.csv"))), None)
    if f is None:
        raise FileNotFoundError(f"tracked_eddy_{name}.csv not found under {DIR_TRACKS}")
    tr = pd.read_csv(f, parse_dates=["date"])
    return tr, f.parent


def center_at(tr, times):
    """TOEddies center at given times: daily centers at 12:00, running mean, linear interp."""
    t = tr.date + pd.Timedelta(hours=12)
    lon = tr.lon.rolling(CENTER_SMOOTH_DAYS, center=True, min_periods=1).mean().values
    lat = tr.lat.rolling(CENTER_SMOOTH_DAYS, center=True, min_periods=1).mean().values
    tq = pd.to_datetime(times).values.astype("datetime64[s]").astype(float)
    tt = t.values.astype("datetime64[s]").astype(float)
    return np.interp(tq, tt, lon), np.interp(tq, tt, lat)


def rmax_at(tr, t0, t1):
    m = (tr.date >= pd.Timestamp(t0).normalize()) & (tr.date <= pd.Timestamp(t1))
    return float(tr.loc[m, "r_maxspeed_km"].mean())


def eddy_contours(tr, day, eddy_type):
    """Max-speed and outermost contours of the tracked eddy on that day (lon/lat arrays)."""
    if gpd is None:
        return None, None
    row = tr.iloc[(tr.date - pd.Timestamp(day).normalize()).abs().argmin()]
    folder = DIR_EDDIES.get(row.source, DIR_EDDIES["TOEDDIES"])
    kind = "anticyclones" if eddy_type == 1 else "cyclones"
    d = row.date
    res = []
    for part in ("maxspeed", "outcontour"):
        xy = None
        for tag in (d.strftime("%Y%m%d"), d.strftime("%Y-%m-%d")):
            for f in sorted(glob.glob(str(folder / f"*{tag}*{part}*"))):
                if (kind == "anticyclones") != ("anticyclon" in Path(f).name.lower()):
                    continue
                g = gpd.read_file(f).geometry
                i = int(row.index_in_file)
                if i < len(g) and g.iloc[i] is not None and not g.iloc[i].is_empty:
                    geom = g.iloc[i]
                    if geom.geom_type.startswith("Multi"):
                        geom = max(geom.geoms, key=lambda q: q.length)
                    c = geom.exterior.coords if geom.geom_type == "Polygon" else geom.coords
                    xy = np.asarray(c)[:, :2]
                break
            if xy is not None:
                break
        res.append(xy)
    return res


def adt_day(day):
    d = pd.Timestamp(day)
    tags = (d.strftime("%Y-%m-%d"), d.strftime("%Y%m%d"))
    f = next((f for f in sorted(glob.glob(str(DIR_SSH / "*.nc")))
              if any(t in Path(f).name for t in tags)), None)
    if f is None:
        return None
    ds = xr.open_dataset(f)
    ds = ds.rename({k: v for k, v in {"latitude": "lat", "longitude": "lon"}.items()
                    if k in ds.variables})
    a = ds["adt"]
    if "time" in a.dims:
        a = a.isel(time=0)
    return a.load()


# =============================================================================
# Plot helpers
# =============================================================================
def vlim_of(*arrs):
    v = np.concatenate([np.abs(np.asarray(a)).ravel() for a in arrs])
    v = v[np.isfinite(v)]
    return max(0.1, np.round(np.nanpercentile(v, 98) * 10) / 10) if v.size else 0.5


def draw_section(ax, xg, z, V, vlim, xlabel, title):
    levels = np.linspace(-vlim, vlim, 21)
    Vm = np.ma.masked_invalid(V.T)
    cf = ax.contourf(xg, z, Vm, levels=levels, cmap=CMAP, extend="both")
    try:
        cs = ax.contour(xg, z, Vm, levels=np.arange(-2, 2.01, 0.1), colors="k", linewidths=0.4)
        ax.clabel(cs, fmt="%.1f", fontsize=6)
    except Exception:
        pass
    ax.set_ylim(z.max() if np.isfinite(z.max()) else 1000, 0)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Depth (m)")
    ax.set_title(title, loc="left", fontsize=10)
    return cf


def geo_map(ax, extent, tr, day, eddy_type):
    ax.set_extent(extent, crs=crs)
    ax.add_feature(cfeature.LAND, facecolor="0.6", edgecolor="k", zorder=3)
    gl = ax.gridlines(draw_labels=True, lw=0.3, color="0.5", alpha=0.5)
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = gl.ylabel_style = {"size": 7}
    a = adt_day(day)
    if a is not None:
        a = a.sel(lon=slice(extent[0] - 1, extent[1] + 1), lat=slice(extent[2] - 1, extent[3] + 1))
        ax.contour(a.lon, a.lat, a, levels=np.arange(-2, 3, 0.05), colors="0.5",
                   linewidths=0.4, transform=crs, zorder=1)
    ms, oc = eddy_contours(tr, day, eddy_type)
    col = "tab:red" if eddy_type == 1 else "tab:blue"
    if ms is not None:
        ax.plot(ms[:, 0], ms[:, 1], color=col, lw=1.5, transform=crs, zorder=4)
    if oc is not None:
        ax.plot(oc[:, 0], oc[:, 1], color="k", lw=0.8, ls=":", transform=crs, zorder=4)


# =============================================================================
# Section mode
# =============================================================================
def run_section(key, sec, cfg, tr, out_dir, instrument):
    t0, t1 = sec["t0"], sec["t1"]
    tmid = pd.Timestamp(t0) + (pd.Timestamp(t1) - pd.Timestamp(t0)) / 2
    clon, clat = center_at(tr, [tmid])
    clon, clat = float(clon[0]), float(clat[0])
    rmax = rmax_at(tr, t0, t1)

    # ship track defines the line for both instruments (same line -> comparable figures)
    sad = sadcp_window(t0, t1)
    lon0, lat0 = float(np.nanmean(sad.lon)), float(np.nanmean(sad.lat))
    xs, ys = to_xy(sad.lon.values, sad.lat.values, lon0, lat0)
    origin, es, en = fit_line(xs, ys)
    proj_s = lambda x, y: (x - origin[0]) * es[0] + (y - origin[1]) * es[1]
    proj_n = lambda x, y: (x - origin[0]) * en[0] + (y - origin[1]) * en[1]
    cx, cy = to_xy(clon, clat, lon0, lat0)
    s_c, d_c = float(proj_s(cx, cy)), float(proj_n(cx, cy))
    heading = (np.degrees(np.arctan2(es[0], es[1])) + 360) % 360

    if instrument == "SADCP":
        s = proj_s(xs, ys)
        vn = sad.u.values * en[0] + sad.v.values * en[1]
        edges = np.arange(np.floor(s.min()), np.ceil(s.max()) + DS_KM, DS_KM)
        xg, V, _ = bin_columns(s, vn, edges)
        z = sad.z.values
        pts_lon, pts_lat, labels = sad.lon.values, sad.lat.values, None
        dist_line = proj_n(xs, ys)
    else:
        stations = load_ladcp(t0, t1)
        if not stations:
            print(f"  no LADCP station in {key}, skipped")
            return
        z = np.arange(0, ZMAX_LADCP + 1, 16.0)
        U, Vv = ladcp_on_grid(stations, z)
        lo = np.array([q["lon"] for q in stations])
        la = np.array([q["lat"] for q in stations])
        x, y = to_xy(lo, la, lon0, lat0)
        s = proj_s(x, y)
        vn = U * en[0] + Vv * en[1]
        xg = np.arange(np.floor(s.min()) - 2, np.ceil(s.max()) + 2.5, 1.0)
        V = interp_columns(s, vn, xg, LADCP_MAX_GAP_KM)
        pts_lon, pts_lat, labels = lo, la, [q["station"] for q in stations]
        dist_line = proj_n(x, y)
        keepz = np.isfinite(V).any(0)
        if keepz.any():
            z, V = z[: np.where(keepz)[0].max() + 1], V[:, : np.where(keepz)[0].max() + 1]

    vlim = cfg.get("vlim") or vlim_of(V)
    fig = plt.figure(figsize=(15, 6))
    gs = GridSpec(1, 2, width_ratios=[1, 2.3], wspace=0.15, figure=fig)
    axm = fig.add_subplot(gs[0], projection=crs)
    lon_all = np.r_[sad.lon.values, clon]
    lat_all = np.r_[sad.lat.values, clat]
    pad = max(0.5, rmax / 111 * 1.3)
    extent = [np.nanmin(lon_all) - pad, np.nanmax(lon_all) + pad,
              np.nanmin(lat_all) - pad, np.nanmax(lat_all) + pad]
    geo_map(axm, extent, tr, tmid, cfg["eddy_type"])
    axm.plot(sad.lon, sad.lat, "-", color="0.3", lw=1, transform=crs, zorder=5)
    axm.scatter(pts_lon, pts_lat, s=8 if labels is None else 30,
                c="k" if labels is None else "lime", edgecolors="k", lw=0.3,
                transform=crs, zorder=6, marker="." if labels is None else "^")
    if labels:
        for lo_, la_, lab in zip(pts_lon, pts_lat, labels):
            axm.text(lo_ + 0.03, la_ + 0.03, lab, fontsize=6, transform=crs, zorder=7)
    # fitted line and direction of positive orthogonal velocity
    smin, smax = np.nanmin(proj_s(xs, ys)), np.nanmax(proj_s(xs, ys))
    lx = origin[0] + np.array([smin, smax]) * es[0]
    ly = origin[1] + np.array([smin, smax]) * es[1]
    llon = lon0 + np.rad2deg(lx / (R_KM * np.cos(np.deg2rad(lat0))))
    llat = lat0 + np.rad2deg(ly / R_KM)
    tf = crs._as_mpl_transform(axm)
    axm.annotate("", xy=(llon[1], llat[1]), xytext=(llon[0], llat[0]), xycoords=tf,
                 textcoords=tf, arrowprops=dict(arrowstyle="->", color="m", lw=1.5), zorder=8)
    mx, my = llon.mean(), llat.mean()
    nlen = (smax - smin) * 0.15
    axm.annotate("+v$_\\perp$", xy=(mx + np.rad2deg(en[0] * nlen / (R_KM * np.cos(np.deg2rad(lat0)))),
                                     my + np.rad2deg(en[1] * nlen / R_KM)),
                 xytext=(mx, my), xycoords=tf, textcoords=tf,
                 arrowprops=dict(arrowstyle="->", color="m"), color="m", fontsize=8, zorder=8)
    axm.plot(clon, clat, "x", color="k", ms=8, mew=2, transform=crs, zorder=9)
    axm.set_title(f"{cfg['name']} — {key}\nline heading {heading:.0f}°, "
                  f"center {abs(d_c):.0f} km from line", loc="left", fontsize=9)
    axm.legend(handles=[
        mlines.Line2D([], [], color="tab:red" if cfg["eddy_type"] == 1 else "tab:blue",
                      label=f"Max-speed contour {tmid:%m-%d}"),
        mlines.Line2D([], [], color="k", ls=":", label="Outermost contour"),
        mlines.Line2D([], [], color="0.3", label="Ship track"),
        mlines.Line2D([], [], color="k", marker="x", ls="", label="TOEddies center"),
        mlines.Line2D([], [], color="m", label="Section axis")], fontsize=6, loc="lower left")

    ax = fig.add_subplot(gs[1])
    cf = draw_section(ax, xg, z, V, vlim, "Distance along section (km)",
                      f"{instrument}: velocity orthogonal to the section "
                      f"(+ to the left of travel, i.e. towards {(heading - 90) % 360:.0f}°)")
    ax.axvline(s_c, color="k", ls="--", lw=1)
    ax.text(s_c, 0, " center", va="bottom", fontsize=7)
    if np.isfinite(rmax) and rmax > abs(d_c):
        half = np.sqrt(rmax ** 2 - d_c ** 2)
        for xx in (s_c - half, s_c + half):
            ax.axvline(xx, color="tab:red" if cfg["eddy_type"] == 1 else "tab:blue", lw=1.5)
        ax.text(s_c + half, 0, " R$_{max}$", va="bottom", fontsize=7)
    if labels:
        for sv, lab in zip(s, labels):
            ax.annotate(lab, (sv, 0), xytext=(0, 2), textcoords="offset points", fontsize=6,
                        ha="center", va="bottom", rotation=90, annotation_clip=False)
            ax.axvline(sv, color="0.3", lw=0.4, ls=":")
    cb = fig.colorbar(cf, ax=ax, pad=0.01)
    cb.set_label("v$_\\perp$ (m s$^{-1}$)")
    fig.text(0.99, 0.01, f"{pd.Timestamp(t0):%Y-%m-%d %H:%M} → {pd.Timestamp(t1):%Y-%m-%d %H:%M}"
             f" | R$_{{max}}$ (TOEddies) {rmax:.0f} km | samples within "
             f"{np.nanmax(np.abs(dist_line)):.0f} km of the section axis", ha="right", fontsize=7)
    out = out_dir / f"{cfg['name']}_{key}_{instrument}_orthogonal_velocity.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {out}")


# =============================================================================
# Radial mode
# =============================================================================
_FITTED = {}


def run_radial(key, sec, cfg, tr, out_dir, instrument):
    t0, t1 = sec["t0"], sec["t1"]
    rlim = sec.get("r_limit_km", 200.0)
    rmax = rmax_at(tr, t0, t1)
    tmid = pd.Timestamp(t0) + (pd.Timestamp(t1) - pd.Timestamp(t0)) / 2

    def eddy_frame(times, lon, lat):
        clon, clat = center_at(tr, times)
        x = np.deg2rad(lon - clon) * R_KM * np.cos(np.deg2rad(clat))
        y = np.deg2rad(lat - clat) * R_KM
        return x, y

    # center refinement from SADCP (done once, reused for the LADCP figure)
    if (cfg["name"], key) not in _FITTED:
        sad = sadcp_window(t0, t1)
        x, y = eddy_frame(sad.time.values, sad.lon.values, sad.lat.values)
        lay = (sad.z >= FIT_LAYER[0]) & (sad.z <= FIT_LAYER[1])
        ul = sad.u.sel(z=lay).mean("z").values
        vl = sad.v.sel(z=lay).mean("z").values
        if FIT_CENTER:
            dx, dy, cost = fit_center(x, y, ul, vl, r_max=1.5 * rmax)
            _, vt0, vr0 = polar(x, y, ul, vl)
            m0 = np.hypot(x, y) <= 1.5 * rmax
            c0 = np.nansum(vr0[m0] ** 2) / np.nansum(vr0[m0] ** 2 + vt0[m0] ** 2)
            print(f"  center refined by ({dx:+.0f}, {dy:+.0f}) km east/north of TOEddies; "
                  f"radial energy fraction {c0:.2f} -> {cost:.2f}")
        else:
            dx = dy = 0.0
        _FITTED[(cfg["name"], key)] = (dx, dy, sad)
    dx, dy, sad = _FITTED[(cfg["name"], key)]

    if instrument == "SADCP":
        x, y = eddy_frame(sad.time.values, sad.lon.values, sad.lat.values)
        x, y = x - dx, y - dy
        r, vt, _ = polar(x, y, sad.u.values, sad.v.values)
        edges = np.arange(0, rlim + R_BIN_KM, R_BIN_KM)
        rg, V, cnt = bin_columns(r, vt, edges, nmin=3)
        z = sad.z.values
        labels, ptimes = None, sad.time.values
    else:
        stations = load_ladcp(t0, t1)
        if not stations:
            print(f"  no LADCP station in {key}, skipped")
            return
        z = np.arange(0, ZMAX_LADCP + 1, 16.0)
        U, Vv = ladcp_on_grid(stations, z)
        times = np.array([q["time"] for q in stations], "datetime64[ns]")
        lo = np.array([q["lon"] for q in stations])
        la = np.array([q["lat"] for q in stations])
        x, y = eddy_frame(times, lo, la)
        x, y = x - dx, y - dy
        r, vt, _ = polar(x, y, U, Vv)
        rg = np.arange(0, rlim + 1, 2.0)
        # average stations at nearly the same radius (repeat stations), then interpolate
        rb, Vb, cnt = bin_columns(r, vt, np.arange(0, rlim + 5.1, 5.0))
        V = interp_columns(rb[cnt > 0], Vb[cnt > 0], rg, LADCP_MAX_GAP_KM)
        labels, ptimes = [q["station"] for q in stations], times
        keepz = np.isfinite(V).any(0)
        if keepz.any():
            z, V = z[: np.where(keepz)[0].max() + 1], V[:, : np.where(keepz)[0].max() + 1]

    vlim = cfg.get("vlim") or vlim_of(V)
    sign = "anticlockwise" if cfg["eddy_type"] == 1 else "clockwise"
    fig = plt.figure(figsize=(16, 9))
    gs = GridSpec(2, 2, width_ratios=[1, 2], height_ratios=[1, 1.4], hspace=0.3, wspace=0.18,
                  figure=fig)

    # (a) coverage in the eddy frame
    axf = fig.add_subplot(gs[:, 0])
    tnum = (pd.to_datetime(ptimes) - pd.Timestamp(t0)) / pd.Timedelta(days=1)
    sc = axf.scatter(x, y, c=tnum, cmap="viridis", s=6 if labels is None else 40,
                     marker="." if labels is None else "^", edgecolors="none" if labels is None else "k")
    if labels:
        for xx, yy, lab in zip(x, y, labels):
            axf.text(xx + 3, yy + 3, lab, fontsize=6)
    th = np.linspace(0, 2 * np.pi, 200)
    col = "tab:red" if cfg["eddy_type"] == 1 else "tab:blue"
    axf.plot(rmax * np.cos(th), rmax * np.sin(th), color=col, lw=1.5, label=f"R$_{{max}}$ {rmax:.0f} km")
    for rr in np.arange(50, rlim + 1, 50):
        axf.plot(rr * np.cos(th), rr * np.sin(th), color="0.7", lw=0.5)
    axf.plot(0, 0, "+", color="k", ms=12, mew=2, label="refined center")
    axf.plot(-dx, -dy, "x", color="0.4", ms=8, mew=1.5, label="TOEddies center")
    axf.set_aspect("equal")
    axf.set_xlim(-rlim, rlim)
    axf.set_ylim(-rlim, rlim)
    axf.set_xlabel("East of eddy center (km)")
    axf.set_ylabel("North of eddy center (km)")
    axf.legend(fontsize=7, loc="lower left")
    axf.set_title(f"(a) {instrument} positions in the eddy frame", loc="left", fontsize=10)
    cbt = fig.colorbar(sc, ax=axf, orientation="horizontal", pad=0.08, shrink=0.8)
    cbt.set_label(f"days since {pd.Timestamp(t0):%Y-%m-%d}")

    # (b) v_theta(r) at a few depths
    axp = fig.add_subplot(gs[0, 1])
    cmap = plt.get_cmap("viridis")
    for k, zz in enumerate(PROFILE_DEPTHS):
        iz = np.argmin(np.abs(z - zz))
        if abs(z[iz] - zz) > 30:
            continue
        axp.plot(rg, V[:, iz], "-o", ms=2.5, lw=1.2, color=cmap(k / max(1, len(PROFILE_DEPTHS) - 1)),
                 label=f"{z[iz]:.0f} m")
    axp.axhline(0, color="k", lw=0.6)
    axp.axvline(rmax, color=col, lw=1.2)
    axp.set_xlim(0, rlim)
    axp.set_ylabel("v$_\\theta$ (m s$^{-1}$)")
    axp.legend(fontsize=7, ncol=len(PROFILE_DEPTHS))
    axp.grid(alpha=0.3)
    axp.set_title(f"(b) Azimuthal velocity vs radius",
                  loc="left", fontsize=10)

    # (c) v_theta(r, z)
    ax = fig.add_subplot(gs[1, 1], sharex=axp)
    cf = draw_section(ax, rg, z, V, vlim, "Distance to eddy center (km)",
                      f"(c) {instrument}: velocity orthogonal to the radius, v$_\\theta$(r, z)")
    ax.axvline(rmax, color=col, lw=1.5)
    if labels:
        for rv, lab in zip(r, labels):
            ax.annotate(lab, (rv, 0), xytext=(0, 2), textcoords="offset points", fontsize=6,
                        ha="center", va="bottom", rotation=90, annotation_clip=False)
            ax.axvline(rv, color="0.3", lw=0.4, ls=":")
    cb = fig.colorbar(cf, ax=ax, pad=0.01)
    cb.set_label("v$_\\theta$ (m s$^{-1}$)")
    fig.suptitle(f"{cfg['name']} — {key}: {pd.Timestamp(t0):%Y-%m-%d} → {pd.Timestamp(t1):%Y-%m-%d}, "
                 f"eddy frame (TOEddies center smoothed over {CENTER_SMOOTH_DAYS} d, "
                 f"refined by ({dx:+.0f}, {dy:+.0f}) km)", fontsize=11)
    out = out_dir / f"{cfg['name']}_{key}_{instrument}_orthogonal_velocity.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {out}")



In [14]:
# =============================================================================
# Radial mode — ONE figure: (a) eddy frame, (b)/(c) SADCP, (d)/(e) LADCP
# Add this below the existing "Radial mode" section (it reuses its helpers).
# =============================================================================
LABEL_MIN_DR_KM = 4.0      # don't write two station labels closer than this (km)


def _eddy_frame(tr, times, lon, lat):
    clon, clat = center_at(tr, times)
    x = np.deg2rad(lon - clon) * R_KM * np.cos(np.deg2rad(clat))
    y = np.deg2rad(lat - clat) * R_KM
    return x, y


def _refined_center(key, cfg, tr, t0, t1, rmax):
    """Center refinement from SADCP, computed once per (eddy, section) and cached."""
    if (cfg["name"], key) not in _FITTED:
        sad = sadcp_window(t0, t1)
        x, y = _eddy_frame(tr, sad.time.values, sad.lon.values, sad.lat.values)
        lay = (sad.z >= FIT_LAYER[0]) & (sad.z <= FIT_LAYER[1])
        ul = sad.u.sel(z=lay).mean("z").values
        vl = sad.v.sel(z=lay).mean("z").values
        if FIT_CENTER:
            dx, dy, cost = fit_center(x, y, ul, vl, r_max=1.5 * rmax)
            _, vt0, vr0 = polar(x, y, ul, vl)
            m0 = np.hypot(x, y) <= 1.5 * rmax
            c0 = np.nansum(vr0[m0] ** 2) / np.nansum(vr0[m0] ** 2 + vt0[m0] ** 2)
            print(f"  center refined by ({dx:+.0f}, {dy:+.0f}) km east/north of TOEddies; "
                  f"radial energy fraction {c0:.2f} -> {cost:.2f}")
        else:
            dx = dy = 0.0
        _FITTED[(cfg["name"], key)] = (dx, dy, sad)
    return _FITTED[(cfg["name"], key)]


def _radial_fields(instrument, tr, t0, t1, rlim, dx, dy, sad):
    """v_theta(r, z) for one instrument in the (refined) eddy frame, or None."""
    if instrument == "SADCP":
        x, y = _eddy_frame(tr, sad.time.values, sad.lon.values, sad.lat.values)
        x, y = x - dx, y - dy
        r, vt, _ = polar(x, y, sad.u.values, sad.v.values)
        rg, V, _ = bin_columns(r, vt, np.arange(0, rlim + R_BIN_KM, R_BIN_KM), nmin=3)
        return dict(x=x, y=y, r=r, rg=rg, V=V, z=sad.z.values, labels=None,
                    times=sad.time.values)

    stations = load_ladcp(t0, t1)
    if not stations:
        return None
    z = np.arange(0, ZMAX_LADCP + 1, 16.0)
    U, Vv = ladcp_on_grid(stations, z)
    times = np.array([q["time"] for q in stations], "datetime64[ns]")
    lo = np.array([q["lon"] for q in stations])
    la = np.array([q["lat"] for q in stations])
    x, y = _eddy_frame(tr, times, lo, la)
    x, y = x - dx, y - dy
    r, vt, _ = polar(x, y, U, Vv)
    rg = np.arange(0, rlim + 1, 2.0)
    rb, Vb, cnt = bin_columns(r, vt, np.arange(0, rlim + 5.1, 5.0))   # merge repeat stations
    V = interp_columns(rb[cnt > 0], Vb[cnt > 0], rg, LADCP_MAX_GAP_KM)
    keepz = np.isfinite(V).any(0)
    if keepz.any():
        kmax = np.where(keepz)[0].max() + 1
        z, V = z[:kmax], V[:, :kmax]
    return dict(x=x, y=y, r=r, rg=rg, V=V, z=z,
                labels=[q["station"] for q in stations], times=times)


def _profiles(ax, D, rmax, col, rlim, title, markers=True):
    cmap = plt.get_cmap("viridis")
    for k, zz in enumerate(PROFILE_DEPTHS):
        iz = np.argmin(np.abs(D["z"] - zz))
        if abs(D["z"][iz] - zz) > 30:
            continue
        ax.plot(D["rg"], D["V"][:, iz], "-o" if markers else "-", ms=2.5, lw=1.3,
                color=cmap(k / max(1, len(PROFILE_DEPTHS) - 1)), label=f"{D['z'][iz]:.0f} m")
    ax.axhline(0, color="k", lw=0.6)
    ax.axvline(rmax, color=col, lw=1.2)
    ax.set_xlim(0, rlim)
    ax.set_ylabel("v$_\\theta$ (m s$^{-1}$)")
    ax.legend(fontsize=7, ncol=len(PROFILE_DEPTHS), loc="upper left")
    ax.grid(alpha=0.3)
    ax.set_title(title, loc="left", fontsize=10)
    plt.setp(ax.get_xticklabels(), visible=False)


def _station_marks(ax, r, labels):
    """Dotted line at every station; labels thinned so they don't pile up."""
    last = -np.inf
    for i in np.argsort(r):
        ax.axvline(r[i], color="0.3", lw=0.4, ls=":")
        if r[i] - last >= LABEL_MIN_DR_KM:
            ax.annotate(labels[i], (r[i], 0), xytext=(0, 2), textcoords="offset points",
                        fontsize=6, ha="center", va="bottom", rotation=90,
                        annotation_clip=False)
            last = r[i]


def _section(ax, D, vlim, zbot, rmax, col, title):
    cf = draw_section(ax, D["rg"], D["z"], D["V"], vlim, "Distance to eddy center (km)", title)
    ax.set_ylim(zbot, 0)                       # same depth axis for SADCP and LADCP
    ax.axvline(rmax, color=col, lw=1.5)
    if D["labels"]:
        _station_marks(ax, D["r"], D["labels"])
    cax = ax.inset_axes([1.015, 0, 0.022, 1])  # colorbar outside: keeps x-axes aligned
    cb = plt.colorbar(cf, cax=cax)
    cb.set_label("v$_\\theta$ (m s$^{-1}$)")
    return cf


def run_radial_combined(key, sec, cfg, tr, out_dir):
    t0, t1 = sec["t0"], sec["t1"]
    rlim = sec.get("r_limit_km", 200.0)
    rmax = rmax_at(tr, t0, t1)
    col = "tab:red" if cfg["eddy_type"] == 1 else "tab:blue"

    dx, dy, sad = _refined_center(key, cfg, tr, t0, t1, rmax)
    S = _radial_fields("SADCP", tr, t0, t1, rlim, dx, dy, sad)
    L = _radial_fields("LADCP", tr, t0, t1, rlim, dx, dy, sad)
    if L is None:
        print(f"  no LADCP station in {key}: panels (d)/(e) left empty")

    # common colour scale and depth axis so (c) and (e) are directly comparable
    vlim = cfg.get("vlim") or vlim_of(S["V"], *([L["V"]] if L else []))
    zbot = max(np.nanmax(S["z"]), np.nanmax(L["z"]) if L else 0)

    fig = plt.figure(figsize=(24, 10))
    gs = GridSpec(2, 3, width_ratios=[1, 1.45, 1.45], height_ratios=[1, 1.4],
                  hspace=0.12, wspace=0.28, figure=fig)

    # ---------------------------------------------------------- (a) eddy frame
    axf = fig.add_subplot(gs[:, 0])
    ndays = (pd.Timestamp(t1) - pd.Timestamp(t0)) / pd.Timedelta(days=1)
    norm = plt.Normalize(0, ndays)
    tday = lambda t: (pd.to_datetime(t) - pd.Timestamp(t0)) / pd.Timedelta(days=1)
    sc = axf.scatter(S["x"], S["y"], c=tday(S["times"]), cmap="viridis", norm=norm,
                     s=5, marker=".", edgecolors="none", zorder=3)
    if L is not None:
        axf.scatter(L["x"], L["y"], c=tday(L["times"]), cmap="viridis", norm=norm,
                    s=60, marker="^", edgecolors="k", lw=0.7, zorder=4)
        for xx, yy, lab in zip(L["x"], L["y"], L["labels"]):
            axf.text(xx + 3, yy + 3, lab, fontsize=6, zorder=5)
    th = np.linspace(0, 2 * np.pi, 200)
    axf.plot(rmax * np.cos(th), rmax * np.sin(th), color=col, lw=1.5)
    for rr in np.arange(50, rlim + 1, 50):
        axf.plot(rr * np.cos(th), rr * np.sin(th), color="0.75", lw=0.5, zorder=1)
    axf.plot(0, 0, "+", color="k", ms=12, mew=2)
    axf.plot(-dx, -dy, "x", color="0.4", ms=8, mew=1.5)
    axf.set_aspect("equal")
    axf.set_xlim(-rlim, rlim)
    axf.set_ylim(-rlim, rlim)
    axf.set_xlabel("East of eddy center (km)")
    axf.set_ylabel("North of eddy center (km)")
    axf.legend(handles=[
        mlines.Line2D([], [], color=col, lw=1.5, label=f"R$_{{max}}$ {rmax:.0f} km"),
        mlines.Line2D([], [], color="k", marker="+", ms=10, mew=2, ls="", label="refined center"),
        mlines.Line2D([], [], color="0.4", marker="x", ms=7, mew=1.5, ls="", label="TOEddies center"),
        mlines.Line2D([], [], color="0.4", marker=".", ms=6, ls="", label="SADCP"),
        mlines.Line2D([], [], color="0.4", marker="^", mec="k", ms=8, ls="", label="LADCP stations"),
    ], fontsize=7, loc="lower left", framealpha=0.9)
    axf.set_title("(a) SADCP and LADCP positions in the eddy frame", loc="left", fontsize=10)
    cbt = fig.colorbar(sc, ax=axf, orientation="horizontal", pad=0.08, shrink=0.8)
    cbt.set_label(f"days since {pd.Timestamp(t0):%Y-%m-%d}")

    # ------------------------------------------------------- (b)/(c) SADCP
    ax_b = fig.add_subplot(gs[0, 1])
    ax_c = fig.add_subplot(gs[1, 1], sharex=ax_b)
    _profiles(ax_b, S, rmax, col, rlim, "(b) SADCP: azimuthal velocity vs radius")
    _section(ax_c, S, vlim, zbot, rmax, col, "(c) SADCP: v$_\\theta$(r, z)")

    # ------------------------------------------------------- (d)/(e) LADCP
    ax_d = fig.add_subplot(gs[0, 2], sharey=ax_b)
    ax_e = fig.add_subplot(gs[1, 2], sharex=ax_d, sharey=ax_c)
    if L is not None:
        _profiles(ax_d, L, rmax, col, rlim, "(d) LADCP: azimuthal velocity vs radius",
                  markers=False)
        for rv in L["r"]:
            ax_d.axvline(rv, color="0.3", lw=0.4, ls=":")
        _section(ax_e, L, vlim, zbot, rmax, col, "(e) LADCP: v$_\\theta$(r, z)")
    else:
        for ax, t in ((ax_d, "(d) LADCP"), (ax_e, "(e) LADCP")):
            ax.text(0.5, 0.5, "no LADCP station in this window", ha="center",
                    va="center", transform=ax.transAxes, color="0.4")
            ax.set_title(t, loc="left", fontsize=10)
        ax_e.set_xlim(0, rlim)
        ax_e.set_ylim(zbot, 0)

    fig.suptitle(f"{cfg['name']} — {key}: {pd.Timestamp(t0):%Y-%m-%d} → {pd.Timestamp(t1):%Y-%m-%d}, "
                 f"eddy frame (TOEddies center smoothed over {CENTER_SMOOTH_DAYS} d, "
                 f"refined by ({dx:+.0f}, {dy:+.0f}) km)", fontsize=12, y=0.97)
    out = out_dir / f"{cfg['name']}_{key}_SADCP_LADCP_orthogonal_velocity.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {out}")

In [15]:
# =============================================================================
# Radial mode — ONE figure: (a) eddy frame, (b)/(c) SADCP, (d)/(e) LADCP
# Add this below the existing "Radial mode" section (it reuses its helpers).
# =============================================================================
LABEL_MIN_DR_KM = 4.0      # don't write two station labels closer than this (km)


def _eddy_frame(tr, times, lon, lat):
    clon, clat = center_at(tr, times)
    x = np.deg2rad(lon - clon) * R_KM * np.cos(np.deg2rad(clat))
    y = np.deg2rad(lat - clat) * R_KM
    return x, y


def _refined_center(key, cfg, tr, t0, t1, rmax):
    """Center refinement from SADCP, computed once per (eddy, section) and cached."""
    if (cfg["name"], key) not in _FITTED:
        sad = sadcp_window(t0, t1)
        x, y = _eddy_frame(tr, sad.time.values, sad.lon.values, sad.lat.values)
        lay = (sad.z >= FIT_LAYER[0]) & (sad.z <= FIT_LAYER[1])
        ul = sad.u.sel(z=lay).mean("z").values
        vl = sad.v.sel(z=lay).mean("z").values
        if FIT_CENTER:
            dx, dy, cost = fit_center(x, y, ul, vl, r_max=1.5 * rmax)
            _, vt0, vr0 = polar(x, y, ul, vl)
            m0 = np.hypot(x, y) <= 1.5 * rmax
            c0 = np.nansum(vr0[m0] ** 2) / np.nansum(vr0[m0] ** 2 + vt0[m0] ** 2)
            print(f"  center refined by ({dx:+.0f}, {dy:+.0f}) km east/north of TOEddies; "
                  f"radial energy fraction {c0:.2f} -> {cost:.2f}")
        else:
            dx = dy = 0.0
        _FITTED[(cfg["name"], key)] = (dx, dy, sad)
    return _FITTED[(cfg["name"], key)]


def _radial_fields(instrument, tr, t0, t1, rlim, dx, dy, sad):
    """v_theta(r, z) for one instrument in the (refined) eddy frame, or None."""
    if instrument == "SADCP":
        x, y = _eddy_frame(tr, sad.time.values, sad.lon.values, sad.lat.values)
        x, y = x - dx, y - dy
        r, vt, _ = polar(x, y, sad.u.values, sad.v.values)
        rg, V, _ = bin_columns(r, vt, np.arange(0, rlim + R_BIN_KM, R_BIN_KM), nmin=3)
        return dict(x=x, y=y, r=r, rg=rg, V=V, z=sad.z.values, labels=None,
                    times=sad.time.values)

    stations = load_ladcp(t0, t1)
    if not stations:
        return None
    z = np.arange(0, ZMAX_LADCP + 1, 16.0)
    U, Vv = ladcp_on_grid(stations, z)
    times = np.array([q["time"] for q in stations], "datetime64[ns]")
    lo = np.array([q["lon"] for q in stations])
    la = np.array([q["lat"] for q in stations])
    x, y = _eddy_frame(tr, times, lo, la)
    x, y = x - dx, y - dy
    r, vt, _ = polar(x, y, U, Vv)
    rg = np.arange(0, rlim + 1, 2.0)
    rb, Vb, cnt = bin_columns(r, vt, np.arange(0, rlim + 5.1, 5.0))   # merge repeat stations
    V = interp_columns(rb[cnt > 0], Vb[cnt > 0], rg, LADCP_MAX_GAP_KM)
    keepz = np.isfinite(V).any(0)
    if keepz.any():
        kmax = np.where(keepz)[0].max() + 1
        z, V = z[:kmax], V[:, :kmax]
    return dict(x=x, y=y, r=r, rg=rg, V=V, z=z,
                labels=[q["station"] for q in stations], times=times)


def _profiles(ax, D, rmax, col, rlim, title, markers=True):
    cmap = plt.get_cmap("viridis")
    for k, zz in enumerate(PROFILE_DEPTHS):
        iz = np.argmin(np.abs(D["z"] - zz))
        if abs(D["z"][iz] - zz) > 30:
            continue
        ax.plot(D["rg"], D["V"][:, iz], "-o" if markers else "-", ms=2.5, lw=1.3,
                color=cmap(k / max(1, len(PROFILE_DEPTHS) - 1)), label=f"{D['z'][iz]:.0f} m")
    ax.axhline(0, color="k", lw=0.6)
    ax.axvline(rmax, color=col, lw=1.2)
    ax.set_xlim(0, rlim)
    ax.set_ylabel("v$_\\theta$ (m s$^{-1}$)")
    ax.legend(fontsize=7, ncol=len(PROFILE_DEPTHS), loc="upper left")
    ax.grid(alpha=0.3)
    ax.set_title(title, loc="left", fontsize=10)
    plt.setp(ax.get_xticklabels(), visible=False)


def _station_marks(ax, r, labels):
    """Dotted line at every station; labels thinned so they don't pile up."""
    last = -np.inf
    for i in np.argsort(r):
        ax.axvline(r[i], color="0.3", lw=0.4, ls=":")
        if r[i] - last >= LABEL_MIN_DR_KM:
            ax.annotate(labels[i], (r[i], 0), xytext=(0, 2), textcoords="offset points",
                        fontsize=6, ha="center", va="bottom", rotation=90,
                        annotation_clip=False)
            last = r[i]


def _section(ax, D, vlim, zbot, rmax, col, title):
    cf = draw_section(ax, D["rg"], D["z"], D["V"], vlim, "Distance to eddy center (km)", title)
    ax.set_ylim(zbot, 0)                       # same depth axis for SADCP and LADCP
    ax.axvline(rmax, color=col, lw=1.5)
    if D["labels"]:
        _station_marks(ax, D["r"], D["labels"])
    cax = ax.inset_axes([1.015, 0, 0.022, 1])  # colorbar outside: keeps x-axes aligned
    cb = plt.colorbar(cf, cax=cax)
    cb.set_label("v$_\\theta$ (m s$^{-1}$)")
    return cf


def run_radial_combined(key, sec, cfg, tr, out_dir):
    t0, t1 = sec["t0"], sec["t1"]
    rlim = sec.get("r_limit_km", 200.0)
    rmax = rmax_at(tr, t0, t1)
    col = "tab:red" if cfg["eddy_type"] == 1 else "tab:blue"

    dx, dy, sad = _refined_center(key, cfg, tr, t0, t1, rmax)
    S = _radial_fields("SADCP", tr, t0, t1, rlim, dx, dy, sad)
    L = _radial_fields("LADCP", tr, t0, t1, rlim, dx, dy, sad)
    if L is None:
        print(f"  no LADCP station in {key}: panels (d)/(e) left empty")

    # common colour scale and depth axis so (c) and (e) are directly comparable
    vlim = cfg.get("vlim") or vlim_of(S["V"], *([L["V"]] if L else []))
    zbot = max(np.nanmax(S["z"]), np.nanmax(L["z"]) if L else 0)

    fig = plt.figure(figsize=(24, 10))
    gs = GridSpec(2, 3, width_ratios=[1, 1.45, 1.45], height_ratios=[1, 1.4],
                  hspace=0.12, wspace=0.28, figure=fig)

    # ---------------------------------------------------------- (a) eddy frame
    axf = fig.add_subplot(gs[:, 0])
    ndays = (pd.Timestamp(t1) - pd.Timestamp(t0)) / pd.Timedelta(days=1)
    norm = plt.Normalize(0, ndays)
    tday = lambda t: (pd.to_datetime(t) - pd.Timestamp(t0)) / pd.Timedelta(days=1)
    sc = axf.scatter(S["x"], S["y"], c=tday(S["times"]), cmap="viridis", norm=norm,
                     s=5, marker=".", edgecolors="none", zorder=3)
    if L is not None:
        axf.scatter(L["x"], L["y"], c=tday(L["times"]), cmap="viridis", norm=norm,
                    s=60, marker="^", edgecolors="k", lw=0.7, zorder=4)
        for xx, yy, lab in zip(L["x"], L["y"], L["labels"]):
            axf.text(xx + 3, yy + 3, lab, fontsize=6, zorder=5)
    th = np.linspace(0, 2 * np.pi, 200)
    axf.plot(rmax * np.cos(th), rmax * np.sin(th), color=col, lw=1.5)
    for rr in np.arange(50, rlim + 1, 50):
        axf.plot(rr * np.cos(th), rr * np.sin(th), color="0.75", lw=0.5, zorder=1)
    axf.plot(0, 0, "+", color="k", ms=12, mew=2)
    axf.plot(-dx, -dy, "x", color="0.4", ms=8, mew=1.5)
    axf.set_aspect("equal")
    axf.set_xlim(-rlim, rlim)
    axf.set_ylim(-rlim, rlim)
    axf.set_xlabel("East of eddy center (km)")
    axf.set_ylabel("North of eddy center (km)")
    axf.legend(handles=[
        mlines.Line2D([], [], color=col, lw=1.5, label=f"R$_{{max}}$ {rmax:.0f} km"),
        mlines.Line2D([], [], color="k", marker="+", ms=10, mew=2, ls="", label="refined center"),
        mlines.Line2D([], [], color="0.4", marker="x", ms=7, mew=1.5, ls="", label="TOEddies center"),
        mlines.Line2D([], [], color="0.4", marker=".", ms=6, ls="", label="SADCP"),
        mlines.Line2D([], [], color="0.4", marker="^", mec="k", ms=8, ls="", label="LADCP stations"),
    ], fontsize=7, loc="lower left", framealpha=0.9)
    axf.set_title("(a) SADCP and LADCP positions in the eddy frame", loc="left", fontsize=10)
    cbt = fig.colorbar(sc, ax=axf, orientation="horizontal", pad=0.08, shrink=0.8)
    cbt.set_label(f"days since {pd.Timestamp(t0):%Y-%m-%d}")

    # ------------------------------------------------------- (b)/(c) SADCP
    ax_b = fig.add_subplot(gs[0, 1])
    ax_c = fig.add_subplot(gs[1, 1], sharex=ax_b)
    _profiles(ax_b, S, rmax, col, rlim, "(b) SADCP: azimuthal velocity vs radius")
    _section(ax_c, S, vlim, zbot, rmax, col, "(c) SADCP: v$_\\theta$(r, z)")

    # ------------------------------------------------------- (d)/(e) LADCP
    ax_d = fig.add_subplot(gs[0, 2], sharey=ax_b)
    ax_e = fig.add_subplot(gs[1, 2], sharex=ax_d, sharey=ax_c)
    if L is not None:
        _profiles(ax_d, L, rmax, col, rlim, "(d) LADCP: azimuthal velocity vs radius",
                  markers=False)
        for rv in L["r"]:
            ax_d.axvline(rv, color="0.3", lw=0.4, ls=":")
        _section(ax_e, L, vlim, zbot, rmax, col, "(e) LADCP: v$_\\theta$(r, z)")
    else:
        for ax, t in ((ax_d, "(d) LADCP"), (ax_e, "(e) LADCP")):
            ax.text(0.5, 0.5, "no LADCP station in this window", ha="center",
                    va="center", transform=ax.transAxes, color="0.4")
            ax.set_title(t, loc="left", fontsize=10)
        ax_e.set_xlim(0, rlim)
        ax_e.set_ylim(zbot, 0)

    fig.suptitle(f"{cfg['name']} — {key}: {pd.Timestamp(t0):%Y-%m-%d} → {pd.Timestamp(t1):%Y-%m-%d}, "
                 f"eddy frame (TOEddies center smoothed over {CENTER_SMOOTH_DAYS} d, "
                 f"refined by ({dx:+.0f}, {dy:+.0f}) km)", fontsize=12, y=0.97)
    out = out_dir / f"{cfg['name']}_{key}_SADCP_LADCP_orthogonal_velocity.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {out}")

In [21]:
# =============================================================================
# CYCLONE — the ship crossed the eddy (almost) through its center
#   1) find that crossing in the SADCP track, using the TOEddies center
#   2) SADCP figure: velocity orthogonal to the crossing
#   3) LADCP figure: every station around the eddy, radial decomposition
#      (+ one figure with each LADCP section in its own frame)
#   Left column of each figure: (a) satellite view, (b) idealized eddy frame
# =============================================================================
CROSS_R_FACTOR = 2.0        # the crossing = part of the track within this x Rmax of the center
CROSS_MIN_STRAIGHT = 0.7    # chord / path length: rejects passes with U-turns
CROSS_MIN_CHORD = 1.0       # x Rmax: the pass must be at least this long
LADCP_R_FACTOR = 2.5        # LADCP stations kept within this x Rmax (unless r_limit_km is set)
SECTION_MAX_GAP_H = 12.0    # consecutive casts closer than this in time ...
SECTION_MAX_DS_KM = 45.0    # ... and in space belong to the same LADCP section
SECTION_MIN_ST = 3          # a section needs at least this many stations
GROUP_COLORS = plt.get_cmap("tab10").colors


def km_to_lonlat(x, y, lon0, lat0):
    lon = lon0 + np.rad2deg(np.asarray(x, float) / (R_KM * np.cos(np.deg2rad(lat0))))
    lat = lat0 + np.rad2deg(np.asarray(y, float) / R_KM)
    return lon, lat


def equiv_radius(lonlat):
    """Radius (km) of the circle with the same area as a contour."""
    if lonlat is None or len(lonlat) < 4:
        return np.nan
    lon0, lat0 = np.nanmean(lonlat[:, 0]), np.nanmean(lonlat[:, 1])
    x, y = to_xy(lonlat[:, 0], lonlat[:, 1], lon0, lat0)
    area = 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
    return float(np.sqrt(area / np.pi))


def refined_center_lonlat(tr, times, dx, dy):
    clon, clat = center_at(tr, times)
    return km_to_lonlat(dx, dy, clon, clat)


def _layer_mean(sad):
    z = sad.z.values
    lay = (z >= FIT_LAYER[0]) & (z <= FIT_LAYER[1])
    with np.errstate(all="ignore"):
        return np.nanmean(sad.u.values[:, lay], 1), np.nanmean(sad.v.values[:, lay], 1)


# ----------------------------------------------------------------------------- step 1
def _in_track(tr, times):
    """True where the eddy is tracked (center_at would otherwise freeze the first/last center)."""
    t = pd.to_datetime(times)
    return np.asarray((t >= tr.date.min()) & (t <= tr.date.max() + pd.Timedelta(days=1)))


def find_crossing(tr, sad, rmax, r_factor=CROSS_R_FACTOR, pick="closest"):
    """Candidate passes of the ship within r_factor x Rmax of the (moving) TOEddies center.
    Returns a table with a 'chosen' column. pick:
      "closest": the straight pass whose line goes closest to the center
      "first":   the first straight pass through the core
      int:       that row of the printed table (index shown in the 'id' column)"""
    x, y = _eddy_frame(tr, sad.time.values, sad.lon.values, sad.lat.values)
    r = np.hypot(x, y)
    inside = (r <= r_factor * rmax) & _in_track(tr, sad.time.values)
    tmin = sad.time.values.astype("datetime64[m]").astype(float)
    gap = np.r_[True, np.diff(tmin) > 60]                        # data gaps split passes
    start = inside & (np.r_[True, ~inside[:-1]] | gap)
    run = np.where(inside, np.cumsum(start), 0)
    rows = []
    for k in np.unique(run[run > 0]):
        i = np.where(run == k)[0]
        if len(i) < 6:
            continue
        xi, yi = x[i], y[i]
        path = np.nansum(np.hypot(np.diff(xi), np.diff(yi)))
        chord = np.hypot(xi[-1] - xi[0], yi[-1] - yi[0])
        o, es, en = fit_line(xi, yi)
        rows.append(dict(t0=pd.Timestamp(sad.time.values[i[0]]),
                         t1=pd.Timestamp(sad.time.values[i[-1]]), n=len(i),
                         r_min_km=float(r[i].min()),
                         d_line_km=float(abs(-o[0] * en[0] - o[1] * en[1])),
                         chord_km=float(chord),
                         straightness=float(chord / path) if path > 0 else 0.0))
    cand = pd.DataFrame(rows)
    if cand.empty:
        raise RuntimeError(f"the ship never came within {r_factor} Rmax of the center")
    ok = ((cand.straightness >= CROSS_MIN_STRAIGHT) & (cand.chord_km >= CROSS_MIN_CHORD * rmax)
          & (cand.r_min_km <= rmax))
    cand.insert(0, "id", cand.index)
    cand["valid"] = ok
    if isinstance(pick, (int, np.integer)):
        best = int(pick)
    else:
        if not ok.any():
            print("  ! no pass is both straight and through the eddy core: taking the closest one")
        pool = cand[ok] if ok.any() else cand
        best = pool.index[0] if pick == "first" else pool.sort_values("d_line_km").index[0]
    cand["chosen"] = cand.index == best
    return cand


def crossing_offset(tr, sad, rmax):
    """Center correction (km east/north of TOEddies) from the SADCP of the crossing:
    minimises the radial kinetic energy in FIT_LAYER (same method as for the anticyclone,
    but the search is limited to 0.75 Rmax since a single line constrains it less)."""
    if not FIT_CENTER:
        return 0.0, 0.0
    x, y = _eddy_frame(tr, sad.time.values, sad.lon.values, sad.lat.values)
    ul, vl = _layer_mean(sad)
    dx, dy, cost = fit_center(x, y, ul, vl, r_max=1.5 * rmax,
                              search=min(FIT_SEARCH_KM, 0.75 * rmax), step=1.0, nmin=6)
    _, vt0, vr0 = polar(x, y, ul, vl)
    m0 = np.hypot(x, y) <= 1.5 * rmax
    c0 = np.nansum(vr0[m0] ** 2) / np.nansum(vr0[m0] ** 2 + vt0[m0] ** 2)
    print(f"  center refined by ({dx:+.0f}, {dy:+.0f}) km east/north of TOEddies; "
          f"radial energy fraction {c0:.2f} -> {cost:.2f}")
    return dx, dy


# ----------------------------------------------------------------------------- step 2
def crossing_fields(tr, sad, dx, dy, rmax):
    """v_perp(s, z) along the crossing, s = distance from the (refined) center along the line.
    Everything is in the moving eddy frame, so the drift of the eddy during the crossing
    does not distort the section."""
    x, y = _eddy_frame(tr, sad.time.values, sad.lon.values, sad.lat.values)
    x, y = x - dx, y - dy
    o, es, en = fit_line(x, y)
    s = (x - o[0]) * es[0] + (y - o[1]) * es[1]
    s_c = -o[0] * es[0] - o[1] * es[1]
    d_c = -o[0] * en[0] - o[1] * en[1]             # center distance from the line (+ = left)
    srel = s - s_c
    vn = sad.u.values * en[0] + sad.v.values * en[1]
    edges = np.arange(np.floor(np.nanmin(srel) / DS_KM) * DS_KM, np.nanmax(srel) + DS_KM, DS_KM)
    xg, V, cnt = bin_columns(srel, vn, edges)
    heading = (np.degrees(np.arctan2(es[0], es[1])) + 360) % 360

    # observed velocity extrema in FIT_LAYER -> observed Rmax, Vmax, zero crossing
    z = sad.z.values
    lay = (z >= FIT_LAYER[0]) & (z <= FIT_LAYER[1])
    with np.errstate(all="ignore"):
        prof = pd.Series(np.nanmean(V[:, lay], 1)).rolling(3, center=True, min_periods=1).mean().values
    near = np.abs(xg) <= CROSS_R_FACTOR * rmax
    obs = dict(rmax=np.nan, vmax=np.nan, s_mid=np.nan)
    if np.isfinite(prof[near]).sum() > 3:
        p = np.where(near, prof, np.nan)
        ip, ineg = np.nanargmax(p), np.nanargmin(p)
        if xg[ip] * xg[ineg] < 0:
            obs = dict(rmax=float(np.hypot((xg[ip] - xg[ineg]) / 2, d_c)),
                       vmax=float((p[ip] - p[ineg]) / 2), s_mid=float((xg[ip] + xg[ineg]) / 2))
    print(f"  crossing: heading {heading:.0f}°, center {abs(d_c):.1f} km from the line; "
          f"observed Rmax {obs['rmax']:.0f} km (TOEddies {rmax:.0f} km), "
          f"Vmax {obs['vmax']:.2f} m/s, extrema midpoint {obs['s_mid']:+.0f} km from the center")
    return dict(x=x, y=y, lon=sad.lon.values, lat=sad.lat.values, times=sad.time.values,
                s=srel, xg=xg, V=V, z=z, d_c=float(d_c), heading=float(heading), es=es, en=en,
                obs=obs)


# ----------------------------------------------------------------------------- step 3
def cyclone_ladcp(tr, t0, t1, dx, dy, r_lim):
    """All LADCP stations within r_lim of the refined center, in the eddy frame.
    Stations are grouped into sections (consecutive casts close in time and space)."""
    stations = load_ladcp(t0, t1)
    if not stations:
        return None
    times = np.array([q["time"] for q in stations], "datetime64[ns]")
    lo = np.array([q["lon"] for q in stations])
    la = np.array([q["lat"] for q in stations])
    x, y = _eddy_frame(tr, times, lo, la)
    x, y = x - dx, y - dy
    r = np.hypot(x, y)
    keep = (r <= r_lim) & _in_track(tr, times)
    if (~keep).any():
        print(f"  LADCP: {(~keep).sum()} station(s) further than {r_lim:.0f} km left out: "
              + ", ".join(q["station"] for q, k in zip(stations, keep) if not k))
    out = dict(lon=lo[~keep], lat=la[~keep], x=x[~keep], y=y[~keep],
               labels=[q["station"] for q, k in zip(stations, keep) if not k])
    o = np.where(keep)[0][np.argsort(times[keep])]
    stations = [stations[i] for i in o]
    times, lo, la, x, y, r = times[o], lo[o], la[o], x[o], y[o], r[o]
    if not stations:
        return None

    grp = np.zeros(len(stations), int)
    for i in range(1, len(stations)):
        dt_h = (times[i] - times[i - 1]) / np.timedelta64(1, "h")
        xx, yy = to_xy(lo[i], la[i], lo[i - 1], la[i - 1])
        grp[i] = grp[i - 1] + int(dt_h > SECTION_MAX_GAP_H or np.hypot(xx, yy) > SECTION_MAX_DS_KM)

    z = np.arange(0, ZMAX_LADCP + 1, 16.0)
    U, Vv = ladcp_on_grid(stations, z)
    _, vt, _ = polar(x, y, U, Vv)
    rg = np.arange(0, r_lim + 1, 2.0)
    rb, Vb, cnt = bin_columns(r, vt, np.arange(0, r_lim + 5.1, 5.0))   # merges repeat stations
    Vc = interp_columns(rb[cnt > 0], Vb[cnt > 0], rg, LADCP_MAX_GAP_KM)
    keepz = np.isfinite(Vc).any(0) | np.isfinite(vt).any(0)
    kmax = np.where(keepz)[0].max() + 1 if keepz.any() else len(z)
    L = dict(labels=[q["station"] for q in stations], times=times, lon=lo, lat=la, x=x, y=y,
             r=r, grp=grp, z=z[:kmax], U=U[:, :kmax], V=Vv[:, :kmax], vt=vt[:, :kmax],
             rg=rg, Vc=Vc[:, :kmax], sec_name=np.array([""] * len(stations), object))
    L["out"] = out
    L["sections"] = ladcp_sections(L)
    return L


def ladcp_sections(L):
    """v_perp along each LADCP section, in the eddy frame (s = 0 at the closest approach)."""
    secs = []
    for g in np.unique(L["grp"]):
        i = np.where(L["grp"] == g)[0]
        if len(i) < SECTION_MIN_ST:
            continue
        o, es, en = fit_line(L["x"][i], L["y"][i])
        s = (L["x"][i] - o[0]) * es[0] + (L["y"][i] - o[1]) * es[1]
        s_c = -o[0] * es[0] - o[1] * es[1]
        d_c = -o[0] * en[0] - o[1] * en[1]
        srel = s - s_c
        vn = L["U"][i] * en[0] + L["V"][i] * en[1]
        xg = np.arange(np.floor(srel.min()) - 2, np.ceil(srel.max()) + 2.5, 1.0)
        name = f"S{len(secs) + 1}"
        L["sec_name"][i] = name
        secs.append(dict(name=name, idx=i, s=srel, xg=xg,
                         V=interp_columns(srel, vn, xg, LADCP_MAX_GAP_KM), d_c=float(d_c),
                         heading=float((np.degrees(np.arctan2(es[0], es[1])) + 360) % 360),
                         labels=[L["labels"][k] for k in i]))
    print(f"  LADCP sections: " + "; ".join(
        f"{q['name']} = {q['labels'][0]}-{q['labels'][-1]} ({len(q['idx'])} st., "
        f"line {abs(q['d_c']):.0f} km from the center)" for q in secs))
    return secs


# ----------------------------------------------------------------------------- plotting
def _group_color(L, i):
    return GROUP_COLORS[int(L["grp"][i]) % len(GROUP_COLORS)]


def _arrow(ax, x, y, color, transform=None, lw=1.5):
    kw = dict(arrowprops=dict(arrowstyle="->", color=color, lw=lw), zorder=8)
    if transform is not None:
        kw.update(xycoords=transform, textcoords=transform)
    n = len(x)
    i0, i1 = int(0.8 * n), n - 1
    ax.annotate("", xy=(x[i1], y[i1]), xytext=(x[i0], y[i0]), **kw)


def _sat_map(fig, cell, C, day, emphasis):
    """(a) True eddy (satellite ADT + TOEddies contours), SADCP crossing and LADCP stations."""
    ax = fig.add_subplot(cell, projection=crs)
    X, L, col = C["X"], C["L"], C["col"]
    clon, clat = refined_center_lonlat(C["tr"], [day], C["dx"], C["dy"])
    tlon, tlat = center_at(C["tr"], [day])
    half = C["rlim"] * 1.05 / 111.0
    lons = np.r_[X["lon"], L["lon"] if L else [], clon - half / np.cos(np.deg2rad(clat)),
                 clon + half / np.cos(np.deg2rad(clat))]
    lats = np.r_[X["lat"], L["lat"] if L else [], clat - half, clat + half]
    extent = [np.nanmin(lons) - 0.1, np.nanmax(lons) + 0.1, np.nanmin(lats) - 0.1, np.nanmax(lats) + 0.1]
    geo_map(ax, extent, C["tr"], day, C["cfg"]["eddy_type"])
    # center drift over the survey
    tt = pd.date_range(C["survey"][0], C["survey"][1], freq="6h")
    dlon, dlat = refined_center_lonlat(C["tr"], tt, C["dx"], C["dy"])
    ax.plot(dlon, dlat, color=col, lw=1, ls="--", transform=crs, zorder=4)
    # ship
    ax.plot(C["sad_all"].lon.values, C["sad_all"].lat.values, color="0.65", lw=0.5,
            transform=crs, zorder=5)
    lw = 2.2 if emphasis == "SADCP" else 1.4
    ax.plot(X["lon"], X["lat"], color="k", lw=lw, transform=crs, zorder=6)
    _arrow(ax, X["lon"], X["lat"], "k", crs._as_mpl_transform(ax), lw)
    if L:
        ms = 40 if emphasis == "LADCP" else 22
        ax.scatter(L["lon"], L["lat"], s=ms, marker="^", c=[_group_color(L, i) for i in range(len(L["lon"]))],
                   edgecolors="k", lw=0.5, transform=crs, zorder=7)
        ax.scatter(L["out"]["lon"], L["out"]["lat"], s=ms, marker="^", facecolors="none",
                   edgecolors="0.4", lw=0.7, transform=crs, zorder=7)
        if emphasis == "LADCP":
            for lo_, la_, lab in zip(np.r_[L["lon"], L["out"]["lon"]], np.r_[L["lat"], L["out"]["lat"]],
                                     L["labels"] + L["out"]["labels"]):
                ax.text(lo_ + 0.02, la_ + 0.02, lab, fontsize=5.5, transform=crs, zorder=8,
                        clip_on=True)
    ax.plot(tlon, tlat, "x", color="0.3", ms=8, mew=1.5, transform=crs, zorder=9)
    ax.plot(clon, clat, "+", color="k", ms=12, mew=2, transform=crs, zorder=9)
    ax.legend(handles=[
        mlines.Line2D([], [], color=col, lw=1.5, label=f"Max-speed contour {pd.Timestamp(day):%m-%d}"),
        mlines.Line2D([], [], color="k", ls=":", lw=0.8, label="Outermost contour"),
        mlines.Line2D([], [], color=col, ls="--", lw=1, label="Center drift (survey)"),
        mlines.Line2D([], [], color="k", lw=2, label="SADCP crossing"),
        mlines.Line2D([], [], color="0.65", lw=1, label="Ship track (survey)"),
        mlines.Line2D([], [], color="0.5", marker="^", mec="k", ls="", label="LADCP stations (used)"),
        mlines.Line2D([], [], color="none", marker="^", mec="0.4", ls="", label="LADCP (beyond r limit)"),
        mlines.Line2D([], [], color="0.3", marker="x", ls="", label="TOEddies center"),
        mlines.Line2D([], [], color="k", marker="+", ms=10, mew=2, ls="", label="Refined center"),
    ], fontsize=6, loc="best", framealpha=0.9)
    ax.set_title(f"(a) Satellite view, {pd.Timestamp(day):%Y-%m-%d} (ADT, TOEddies)",
                 loc="left", fontsize=10)
    return ax


def _frame_map(fig, cell, C, emphasis):
    """(b) Idealized eddy: circles in the frame moving with the refined center."""
    ax = fig.add_subplot(cell)
    X, L, col, rlim = C["X"], C["L"], C["col"], C["rlim"]
    th = np.linspace(0, 2 * np.pi, 200)
    for rr in np.arange(50, rlim + 1, 50):
        ax.plot(rr * np.cos(th), rr * np.sin(th), color="0.85", lw=0.5, zorder=1)
    handles = []
    for rad, kw, lab in ((C["rmax"], dict(color=col, lw=1.6), "R$_{max}$ TOEddies"),
                         (X["obs"]["rmax"], dict(color=col, lw=1.2, ls="--"), "R$_{max}$ SADCP"),
                         (C["r_out"], dict(color="k", lw=0.9, ls=":"), "Outermost contour (equiv.)")):
        if np.isfinite(rad):
            ax.plot(rad * np.cos(th), rad * np.sin(th), zorder=2, **kw)
            handles.append(mlines.Line2D([], [], label=f"{lab} {rad:.0f} km", **kw))
    lw = 2.2 if emphasis == "SADCP" else 1.4
    ax.plot(X["x"], X["y"], color="k", lw=lw, zorder=4)
    _arrow(ax, X["x"], X["y"], "k", lw=lw)
    if L:
        ms = 50 if emphasis == "LADCP" else 25
        ax.scatter(L["x"], L["y"], s=ms, marker="^", c=[_group_color(L, i) for i in range(len(L["x"]))],
                   edgecolors="k", lw=0.5, zorder=5)
        ax.scatter(L["out"]["x"], L["out"]["y"], s=ms, marker="^", facecolors="none",
                   edgecolors="0.4", lw=0.7, zorder=5)
        if np.isfinite(C["r_lim"]):
            ax.plot(C["r_lim"] * np.cos(th), C["r_lim"] * np.sin(th), color="0.5", lw=0.8, ls="-.")
            handles.append(mlines.Line2D([], [], color="0.5", lw=0.8, ls="-.",
                                         label=f"LADCP r limit {C['r_lim']:.0f} km"))
        if emphasis == "LADCP":
            for xx, yy, lab in zip(L["x"], L["y"], L["labels"]):
                ax.text(xx + 2, yy + 2, lab, fontsize=5.5, zorder=6)
            for q in L["sections"]:
                i = q["idx"][-1]
                ax.text(L["x"][i], L["y"][i] - 9, q["name"], fontsize=8, fontweight="bold",
                        color=_group_color(L, i), ha="center", va="top", zorder=6)
    ax.plot(0, 0, "+", color="k", ms=12, mew=2, zorder=7)
    ax.plot(-C["dx"], -C["dy"], "x", color="0.3", ms=8, mew=1.5, zorder=7)
    handles += [mlines.Line2D([], [], color="k", lw=2, label="SADCP crossing"),
                mlines.Line2D([], [], color="0.5", marker="^", mec="k", ls="", label="LADCP stations")]
    ax.legend(handles=handles, fontsize=6, loc="best", framealpha=0.9)
    ax.set_aspect("equal")
    ax.set_xlim(-rlim, rlim)
    ax.set_ylim(-rlim, rlim)
    ax.set_xlabel("East of eddy center (km)")
    ax.set_ylabel("North of eddy center (km)")
    ax.set_title("(b) Idealized eddy, frame moving with the center", loc="left", fontsize=10)
    return ax


def _colorbar_right(ax, cf, label):
    cax = ax.inset_axes([1.015, 0, 0.018, 1])
    cb = plt.colorbar(cf, cax=cax)
    cb.set_label(label)


def _rmax_marks(ax, C, with_obs=True):
    """Vertical lines where the TOEddies (and SADCP) Rmax circles cut the crossing line."""
    d = C["X"]["d_c"]
    out = []
    for rad, kw in ((C["rmax"], dict(color=C["col"], lw=1.4)),
                    (C["X"]["obs"]["rmax"] if with_obs else np.nan, dict(color=C["col"], lw=1, ls="--"))):
        if np.isfinite(rad) and rad > abs(d):
            h = np.sqrt(rad ** 2 - d ** 2)
            for xx in (-h, h):
                ax.axvline(xx, **kw)
            out.append(h)
    ax.axvline(0, color="k", ls="--", lw=0.8)
    return out


def plot_cyclone_sadcp(C, out):
    X = C["X"]
    fig = plt.figure(figsize=(21, 11))
    gs = GridSpec(2, 2, width_ratios=[1, 2.1], height_ratios=[1, 1.25], hspace=0.22, wspace=0.12,
                  figure=fig)
    _sat_map(fig, gs[0, 0], C, C["day_cross"], "SADCP")
    _frame_map(fig, gs[1, 0], C, "SADCP")

    ax_p = fig.add_subplot(gs[0, 1])
    cmap = plt.get_cmap("viridis")
    for k, zz in enumerate(PROFILE_DEPTHS):
        iz = np.argmin(np.abs(X["z"] - zz))
        if abs(X["z"][iz] - zz) > 30:
            continue
        ax_p.plot(X["xg"], X["V"][:, iz], "-o", ms=2.5, lw=1.3,
                  color=cmap(k / max(1, len(PROFILE_DEPTHS) - 1)), label=f"{X['z'][iz]:.0f} m")
    ax_p.axhline(0, color="k", lw=0.6)
    _rmax_marks(ax_p, C)
    ax_p.legend(fontsize=7, ncol=len(PROFILE_DEPTHS), loc="upper left")
    ax_p.grid(alpha=0.3)
    ax_p.set_ylabel("v$_\\perp$ (m s$^{-1}$)")
    o = X["obs"]
    ax_p.text(0.99, 0.04, f"SADCP {FIT_LAYER[0]:.0f}-{FIT_LAYER[1]:.0f} m: R$_{{max}}$ {o['rmax']:.0f} km, "
              f"V$_{{max}}$ {o['vmax']:.2f} m s$^{{-1}}$\nTOEddies: R$_{{max}}$ {C['rmax']:.0f} km",
              transform=ax_p.transAxes, ha="right", va="bottom", fontsize=7,
              bbox=dict(fc="w", ec="0.7", alpha=0.9))
    ax_p.set_title("(c) SADCP: velocity orthogonal to the crossing at selected depths",
                   loc="left", fontsize=10)
    plt.setp(ax_p.get_xticklabels(), visible=False)

    ax_s = fig.add_subplot(gs[1, 1], sharex=ax_p)
    to = (X["heading"] - 90) % 360
    cf = draw_section(ax_s, X["xg"], X["z"], X["V"], C["vlim"],
                      f"Distance from the eddy center along the crossing (km), "
                      f"+ in the direction of travel ({X['heading']:.0f}°)",
                      f"(d) SADCP: v$_\\perp$(s, z), + towards {to:.0f}° (left of travel)")
    ax_s.set_ylim(C["zbot"], 0)
    _rmax_marks(ax_s, C)
    ax_s.set_xlim(np.nanmin(X["xg"]), np.nanmax(X["xg"]))
    _colorbar_right(ax_s, cf, "v$_\\perp$ (m s$^{-1}$)")
    fig.suptitle(f"{C['cfg']['name']} — SADCP crossing {pd.Timestamp(C['cross'][0]):%Y-%m-%d %H:%M} → "
                 f"{pd.Timestamp(C['cross'][1]):%Y-%m-%d %H:%M}, center {abs(X['d_c']):.0f} km from the line, "
                 f"refined by ({C['dx']:+.0f}, {C['dy']:+.0f}) km", fontsize=12, y=0.95)
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {out}")


def plot_cyclone_ladcp(C, out):
    L = C["L"]
    fig = plt.figure(figsize=(21, 11))
    gs = GridSpec(2, 2, width_ratios=[1, 2.1], height_ratios=[1, 1.25], hspace=0.22, wspace=0.12,
                  figure=fig)
    _sat_map(fig, gs[0, 0], C, C["day_ladcp"], "LADCP")
    _frame_map(fig, gs[1, 0], C, "LADCP")

    # (c) every station at each depth (points) + the composite (line)
    ax_p = fig.add_subplot(gs[0, 1])
    cmap = plt.get_cmap("viridis")
    for k, zz in enumerate(PROFILE_DEPTHS):
        iz = np.argmin(np.abs(L["z"] - zz))
        if abs(L["z"][iz] - zz) > 30:
            continue
        c = cmap(k / max(1, len(PROFILE_DEPTHS) - 1))
        ax_p.scatter(L["r"], L["vt"][:, iz], s=14, color=c, edgecolors="k", lw=0.3, zorder=3)
        ax_p.plot(L["rg"], L["Vc"][:, iz], "-", lw=1.2, color=c, label=f"{L['z'][iz]:.0f} m")
    ax_p.axhline(0, color="k", lw=0.6)
    ax_p.axvline(C["rmax"], color=C["col"], lw=1.4)
    if np.isfinite(C["X"]["obs"]["rmax"]):
        ax_p.axvline(C["X"]["obs"]["rmax"], color=C["col"], lw=1, ls="--")
    ax_p.set_xlim(0, C["r_lim"])
    ax_p.legend(fontsize=7, ncol=len(PROFILE_DEPTHS), loc="upper right")
    ax_p.grid(alpha=0.3)
    ax_p.set_ylabel("v$_\\theta$ (m s$^{-1}$)")
    ax_p.set_title("(c) LADCP: azimuthal velocity vs radius (points: stations, lines: composite)",
                   loc="left", fontsize=10)
    plt.setp(ax_p.get_xticklabels(), visible=False)

    ax_s = fig.add_subplot(gs[1, 1], sharex=ax_p)
    cf = draw_section(ax_s, L["rg"], L["z"], L["Vc"], C["vlim"], "Distance to eddy center (km)",
                      "(d) LADCP: v$_\\theta$(r, z), + anticlockwise")
    ax_s.set_ylim(C["zbot"], 0)
    ax_s.axvline(C["rmax"], color=C["col"], lw=1.4)
    if np.isfinite(C["X"]["obs"]["rmax"]):
        ax_s.axvline(C["X"]["obs"]["rmax"], color=C["col"], lw=1, ls="--")
    _station_marks(ax_s, L["r"], L["labels"])
    ax_s.set_title(ax_s.get_title("left"), loc="left", fontsize=10, pad=22)
    ax_s.set_xlim(0, C["r_lim"])
    _colorbar_right(ax_s, cf, "v$_\\theta$ (m s$^{-1}$)")
    t = pd.to_datetime(L["times"])
    fig.suptitle(f"{C['cfg']['name']} — {len(L['labels'])} LADCP stations {t.min():%Y-%m-%d} → "
                 f"{t.max():%Y-%m-%d}, eddy frame (center from the SADCP crossing, "
                 f"refined by ({C['dx']:+.0f}, {C['dy']:+.0f}) km)", fontsize=12, y=0.95)
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {out}")


def plot_ladcp_sections(C, out):
    L, secs = C["L"], C["L"]["sections"]
    if not secs:
        print("  no LADCP section with enough stations")
        return
    ncol = min(2, len(secs))
    nrow = int(np.ceil(len(secs) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(9 * ncol, 4.3 * nrow), squeeze=False,
                            sharey=True, constrained_layout=True)
    for ax, q in zip(axs.ravel(), secs):
        t = pd.to_datetime(L["times"][q["idx"]])
        cf = draw_section(ax, q["xg"], L["z"], q["V"], C["vlim"],
                          f"Distance along the section from the closest point to the center (km)",
                          f"{q['name']}: st. {q['labels'][0]}–{q['labels'][-1]}, "
                          f"{t.min():%m-%d %H:%M} → {t.max():%m-%d %H:%M}, heading {q['heading']:.0f}°, "
                          f"center {abs(q['d_c']):.0f} km {'left' if q['d_c'] > 0 else 'right'} of the line")
        ax.set_title(ax.get_title("left"), loc="left", fontsize=9, pad=22,
                     color=_group_color(L, q["idx"][0]))
        ax.set_ylim(C["zbot"], 0)
        ax.axvline(0, color="k", ls="--", lw=0.8)
        if C["rmax"] > abs(q["d_c"]):
            h = np.sqrt(C["rmax"] ** 2 - q["d_c"] ** 2)
            for xx in (-h, h):
                ax.axvline(xx, color=C["col"], lw=1.4)
        _station_marks(ax, q["s"], q["labels"])
    for ax in axs.ravel()[len(secs):]:
        ax.set_visible(False)
    cb = fig.colorbar(cf, ax=axs.ravel().tolist(), shrink=0.8, pad=0.01)
    cb.set_label("v$_\\perp$ (m s$^{-1}$), + to the left of the section direction")
    fig.suptitle(f"{C['cfg']['name']} — LADCP sections in the eddy frame "
                 f"(colors as in the maps; colored lines: TOEddies R$_{{max}}$)", fontsize=12)
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {out}")


# ----------------------------------------------------------------------------- driver
def run_cyclone(cfg, tr, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    s0, s1 = pd.Timestamp(cfg["survey"]["t0"]), pd.Timestamp(cfg["survey"]["t1"])
    col = "tab:red" if cfg["eddy_type"] == 1 else "tab:blue"

    # 1) the crossing
    sad_all = sadcp_window(s0, s1)
    rmax_s = rmax_at(tr, s0, s1)
    if cfg.get("crossing"):
        c0, c1 = pd.Timestamp(cfg["crossing"]["t0"]), pd.Timestamp(cfg["crossing"]["t1"])
        print(f"  crossing imposed: {c0} -> {c1}")
    else:
        cand = find_crossing(tr, sad_all, rmax_s, pick=cfg.get("crossing_pick", "closest"))
        print("  candidate passes (distances to the TOEddies center):")
        print(cand.round(2).to_string(index=False))
        best = cand[cand.chosen].iloc[0]
        c0, c1 = best.t0, best.t1
    t = pd.to_datetime(sad_all.time.values)
    sad = sad_all.isel(time=np.where((t >= c0) & (t <= c1))[0])
    rmax = rmax_at(tr, c0, c1)
    dx, dy = crossing_offset(tr, sad, rmax)

    # 2) SADCP along the crossing
    X = crossing_fields(tr, sad, dx, dy, rmax)

    # 3) LADCP around the eddy
    r_lim = cfg.get("r_limit_km") or LADCP_R_FACTOR * rmax
    L = cyclone_ladcp(tr, s0, s1, dx, dy, r_lim)
    if L is None:
        print("  no LADCP station around the eddy")
    else:
        print("  ! LADCP radial decomposition assumes the crossing's center correction "
              "holds for the whole survey")

    day_cross = c0 + (c1 - c0) / 2
    day_ladcp = pd.to_datetime(L["times"]).sort_values()[len(L["times"]) // 2] if L else day_cross
    ms, oc = eddy_contours(tr, day_cross, cfg["eddy_type"])
    arrs = [X["V"]] + ([L["Vc"]] + [q["V"] for q in L["sections"]] if L else [])
    C = dict(cfg=cfg, tr=tr, col=col, rmax=rmax, r_out=equiv_radius(oc), dx=dx, dy=dy,
             sad_all=sad_all, X=X, L=L, survey=(s0, s1), cross=(c0, c1), r_lim=r_lim,
             rlim=max(r_lim, CROSS_R_FACTOR * rmax) * 1.05,
             day_cross=day_cross, day_ladcp=day_ladcp,
             vlim=cfg.get("vlim") or vlim_of(*arrs),
             zbot=max(np.nanmax(X["z"]), np.nanmax(L["z"]) if L else 0))

    tag = f"{cfg['name']}"
    plot_cyclone_sadcp(C, out_dir / f"{tag}_crossing_SADCP_orthogonal_velocity.png")
    # if L:
    #     plot_cyclone_ladcp(C, out_dir / f"{tag}_LADCP_radial_velocity.png")
    #     plot_ladcp_sections(C, out_dir / f"{tag}_LADCP_sections_orthogonal_velocity.png")
    return C


In [45]:
# =============================================================================
# Run
# =============================================================================
for key in RUN:
    cfg = EDDIES[key]
    tr, _ = load_track(cfg["name"])
    
    out_dir = OUT_DIR / cfg["name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {key} ({cfg['name']}) -> {out_dir}")

    s0, s1 = pd.Timestamp(cfg["survey"]["t0"]), pd.Timestamp(cfg["survey"]["t1"])
    col = "tab:red" if cfg["eddy_type"] == 1 else "tab:blue"

    # 1) the crossing
    sad_all = sadcp_window(s0, s1)
    rmax_s = rmax_at(tr, s0, s1)
    cand = find_crossing(tr, sad_all, rmax_s, pick=cfg.get("crossing_pick", "closest"))
    i = cand["r_min_km"].idxmin()
    cand["chosen"] = cand.index == i          # keep the printed table consistent
    best = cand.loc[i]
    print(best)
    c0, c1 = best.t0, best.t1
    t = pd.to_datetime(sad_all.time.values)
    sad = sad_all.isel(time=np.where((t >= c0) & (t <= c1))[0])
    rmax = rmax_at(tr, c0, c1)
    dx, dy = crossing_offset(tr, sad, rmax)

    # 2) SADCP along the crossing
    X = crossing_fields(tr, sad, dx, dy, rmax)


=== CE18 (CE18_20260523) -> /Users/emmagurcan/Documents/France/ENS/Speich/WHIRLS/figures2/CE18_20260523
id                                3
t0              2026-07-10 06:40:00
t1              2026-07-13 04:40:00
n                               421
r_min_km                    9.89998
d_line_km                 13.234649
chord_km                 288.734879
straightness               0.613074
valid                         False
chosen                         True
Name: 3, dtype: object
  center refined by (+15, -9) km east/north of TOEddies; radial energy fraction 0.07 -> 0.04
  crossing: heading 62°, center 28.2 km from the line; observed Rmax 134 km (TOEddies 90 km), Vmax 0.73 m/s, extrema midpoint -6 km from the center


/var/folders/lg/5krm8xnn5xv0smms_psmknp40000gn/T/ipykernel_4814/2350419792.py:65: RuntimeWarning: Mean of empty slice
  out[b] = np.nanmean(vals[m], axis=0)
/var/folders/lg/5krm8xnn5xv0smms_psmknp40000gn/T/ipykernel_4814/3585395459.py:139: RuntimeWarning: Mean of empty slice
  prof = pd.Series(np.nanmean(V[:, lay], 1)).rolling(3, center=True, min_periods=1).mean().values


In [38]:
type(cand["r_min_km"])

pandas.core.series.Series

In [41]:
i = cand["r_min_km"].idxmin()          # pandas: index label of the minimum (= the "id" column)
# or
# i = np.argmin(cand["r_min_km"].values) # numpy: position of the minimum

In [42]:
i

3

In [20]:
cfg

{'name': 'CE18_20260523',
 'eddy_type': -1,
 'strategy': 'crossing',
 'survey': {'t0': '2026-06-28T00:00', 't1': '2026-07-27T00:00'},
 'crossing_pick': 'first',
 'crossing': None,
 'r_limit_km': None}